In [2]:
from pathlib import Path
import json
from tqdm.auto import tqdm

import numpy as np
from sklearn.decomposition import KernelPCA

from data_extractor import DataExtractor


class KernelPCAEmbedder:
    def __init__(
        self,
        train_root,
        test_root,
        output_root,
        n_components=64,
        kernel="rbf",
        gamma=None,
        random_state=42,
    ):
        self.train_root = Path(train_root)
        self.test_root = Path(test_root)
        self.output_root = Path(output_root)

        # Kernel PCA config
        self.n_components = n_components
        self.kernel = kernel
        self.gamma = gamma
        self.random_state = random_state

        self.train_extractor = DataExtractor(self.train_root)
        self.test_extractor = DataExtractor(self.test_root)

        # Kernel PCA model (core)
        self.model = KernelPCA(
            n_components=self.n_components,
            kernel=self.kernel,
            gamma=self.gamma,
            fit_inverse_transform=False,
            eigen_solver="auto",
            random_state=self.random_state,
        )

    def _discover_shard_indices(self, split_root):
        split_root = Path(split_root)
        label_dir = split_root / "labels"

        if not label_dir.exists():
            raise FileNotFoundError(f"Labels folder not found: {label_dir}")

        shard_files = sorted(label_dir.glob("label_shard_*.npz"))

        if not shard_files:
            raise FileNotFoundError(f"No label shards found in: {label_dir}")

        shard_indices = []
        for path in shard_files:
            shard_str = path.stem.split("_")[-1]
            shard_indices.append(int(shard_str))

        return shard_indices

    def _load_split(self, extractor, extract_fn_name, split_root, split_name):
        if not hasattr(extractor, extract_fn_name):
            raise AttributeError(
                f"{extractor.__class__.__name__} has no method '{extract_fn_name}'."
            )

        extract_fn = getattr(extractor, extract_fn_name)
        shard_indices = self._discover_shard_indices(split_root)

        X_parts = []
        y_parts = []

        for shard_idx in tqdm(
            shard_indices,
            total=len(shard_indices),
            desc=f"Loading {split_name} shards",
            unit="shard",
        ):
            X_shard, y_shard = extract_fn(shard_idx)

            if X_shard.shape[0] != y_shard.shape[0]:
                raise ValueError(
                    f"Mismatch in {split_name} shard {shard_idx:03d}: "
                    f"X has {X_shard.shape[0]} samples, y has {y_shard.shape[0]} labels."
                )

            X_parts.append(X_shard)
            y_parts.append(y_shard)

        X = np.concatenate(X_parts, axis=0).astype(np.float32, copy=False)
        y = np.concatenate(y_parts, axis=0).astype(np.int64, copy=False)

        if X.shape[0] != y.shape[0]:
            raise ValueError(
                f"Mismatch after concatenation for {split_name}: "
                f"X has {X.shape[0]} samples, y has {y.shape[0]} labels."
            )

        return X, y

    def load_train_split(self, extract_fn_name):
        return self._load_split(
            self.train_extractor,
            extract_fn_name,
            self.train_root,
            "train",
        )

    def load_test_split(self, extract_fn_name):
        return self._load_split(
            self.test_extractor,
            extract_fn_name,
            self.test_root,
            "test",
        )

    def fit(self, X):
        self.model.fit(X)
        return self

    def transform(self, X):
        Z = self.model.transform(X)
        return np.asarray(Z, dtype=np.float32)

    def fit_transform(self, X):
        Z = self.model.fit_transform(X)
        return np.asarray(Z, dtype=np.float32)

    def _save_npz_array(self, path, array):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, data=array)

    def save_outputs(
        self,
        Z_train,
        Z_test,
        y_train,
        y_test,
        run_name,
        extra_config=None,
    ):
        run_root = self.output_root / run_name
        run_root.mkdir(parents=True, exist_ok=True)

        save_map = {
            "train_embeddings": (run_root / "train_embeddings.npz", Z_train),
            "test_embeddings": (run_root / "test_embeddings.npz", Z_test),
            "train_labels": (run_root / "train_labels.npz", y_train),
            "test_labels": (run_root / "test_labels.npz", y_test),
        }

        # Kernel PCA does NOT expose components_ like RP
        # but we can optionally save eigenvalues/eigenvectors
        if hasattr(self.model, "eigenvalues_"):
            save_map["eigenvalues"] = (
                run_root / "eigenvalues.npz",
                self.model.eigenvalues_,
            )

        if hasattr(self.model, "eigenvectors_"):
            save_map["eigenvectors"] = (
                run_root / "eigenvectors.npz",
                self.model.eigenvectors_,
            )

        for _, (path, array) in tqdm(
            save_map.items(),
            total=len(save_map),
            desc="Saving artifacts",
            unit="file",
        ):
            self._save_npz_array(path, array)

        run_config = {
            "method": "KernelPCA",
            "n_components": int(self.n_components),
            "kernel": self.kernel,
            "gamma": self.gamma,
            "random_state": int(self.random_state),
            "train_root": str(self.train_root),
            "test_root": str(self.test_root),
            "output_root": str(run_root),
        }

        if extra_config is not None:
            run_config.update(extra_config)

        with open(run_root / "run_config.json", "w", encoding="utf-8") as f:
            json.dump(run_config, f, indent=4)

        return run_root

C:\Users\wind2\anaconda3\envs\ecg_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from data_extractor import DataExtractor
from sklearn.decomposition import KernelPCA

In [4]:
kpca_embedder = KernelPCAEmbedder(
    train_root=r"C:\Users\wind2\ECGdata\processed\v1tts\train",
    test_root=r"C:\Users\wind2\ECGdata\processed\v1tts\test",
    output_root=r"C:\Users\wind2\ECGdata\outputs",
    n_components=64,
    kernel="rbf",
    gamma=None,
    random_state=42
)

In [5]:
X_train, y_train = kpca_embedder.load_train_split("TD_extractor")
X_test, y_test = kpca_embedder.load_test_split("TD_extractor")

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

Loading test shards: 100%|████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  8.56shard/s]

(88197, 722) (88197,)
(22050, 722) (22050,)


In [ ]:
## this is the code which crushed the computer due to memeory issues!! 
Z_train = kpca_embedder.fit_transform(X_train)
Z_test = kpca_embedder.transform(X_test)

print(Z_train.shape)
print(Z_test.shape)

In [ ]:
## encounter memory issue.  subsample 
## Kernel PCA was evaluated on a reduced subset due to quadratic memory complexity, 
## whereas Gaussian Random Projection scales efficiently to the full dataset.”

X_train_small = X_train[:5000]
y_train_small = y_train[:5000]

X_test_small = X_test[:2000]
y_test_small = y_test[:2000]

In [ ]:
Z_train = kpca_embedder.model.fit_transform(X_train_small)
Z_test = kpca_embedder.model.transform(X_test_small)

In [ ]:
## save all the outputs into the folder

kpca_embedder.save_outputs(
    Z_train=Z_train,
    Z_test=Z_test,
    y_train=y_train_small,
    y_test=y_test_small,
    run_name="kpca_td_rbf_5000",
    extra_config={
        "method": "KernelPCA",
        "kernel": "rbf",
        "train_size": 5000,
        "test_size": 2000
    }
)

In [ ]:
## option 1:  GaussianRandomProjection baseline ( use GRP method on a small dataset as used in KPCA method)
from sklearn.random_projection import GaussianRandomProjection

grp = GaussianRandomProjection(n_components=64, random_state=42)

Z_train_grp = grp.fit_transform(X_train_small)
Z_test_grp = grp.transform(X_test_small)

In [ ]:
## Option2: replacement , use approx to replace Kernel PCA ---Backup
from sklearn.kernel_approximation import RBFSampler

rff = RBFSampler(gamma=0.01, n_components=256, random_state=42)

Z_train = rff.fit_transform(X_train)
Z_test = rff.transform(X_test)


In [ ]:
## load output back for evaluation (retrieval)
from pathlib import Path
import numpy as np

run_dir = Path(r"C:\Users\wind2\ECGdata\outputs\kpca_td_rbf_5000")

Z_train_kpca = np.load(run_dir / "train_embeddings.npz")["data"]
Z_test_kpca  = np.load(run_dir / "test_embeddings.npz")["data"]

y_train_kpca = np.load(run_dir / "train_labels.npz")["data"]
y_test_kpca  = np.load(run_dir / "test_labels.npz")["data"]

print(Z_train_kpca.shape, Z_test_kpca.shape)

In [ ]:
## retrieval evaluation
# step 1
from sklearn.neighbors import NearestNeighbors
import numpy as np
knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
knn.fit(Z_train_kpca)

In [ ]:
distances, indices = knn.kneighbors(Z_test_kpca)

In [ ]:
## step 2.  majority vote ( k: 1, 5, 10)
def compute_knn_metrics(indices, y_train, y_test, K_values=[1,5,10]):
    results = {}

    for K in K_values:
        preds = []

        for i in range(len(indices)):
            neighbor_labels = y_train[indices[i][:K]]
            pred = np.bincount(neighbor_labels).argmax()
            preds.append(pred)

        preds = np.array(preds)

        acc = (preds == y_test).mean()
        results[K] = acc

    return results

In [ ]:
kpca_results = compute_knn_metrics(indices, y_train_kpca, y_test_kpca)

print(kpca_results)

In [ ]:
## KPCA 64
from sklearn.decomposition import KernelPCA

kpca_64 = KernelPCA(
    n_components=64,
    kernel="rbf",
    gamma=0.01  # optional but recommended to control behavior
)

Z_train_64 = kpca_64.fit_transform(X_train_small)
Z_test_64 = kpca_64.transform(X_test_small)

In [ ]:
## KPCA 128
kpca_128 = KernelPCA(
    n_components=128,
    kernel="rbf",
    gamma=0.01
)

Z_train_128 = kpca_128.fit_transform(X_train_small)
Z_test_128 = kpca_128.transform(X_test_small)

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def knn_eval(Z_train, y_train, Z_test, y_test, K_values=[1,5,10]):
    knn = NearestNeighbors(n_neighbors=max(K_values), metric="euclidean")
    knn.fit(Z_train)

    distances, indices = knn.kneighbors(Z_test)

    results = {}

    for K in K_values:
        preds = []

        for i in range(len(indices)):
            neighbor_labels = y_train[indices[i][:K]]
            pred = np.bincount(neighbor_labels).argmax()
            preds.append(pred)

        preds = np.array(preds)
        results[K] = float((preds == y_test).mean())

    return results

In [ ]:
kpca_64_results = knn_eval(
    Z_train_64, y_train,
    Z_test_64, y_test
)

kpca_64_results

In [ ]:
print(Z_test_64.shape)
print(y_test.shape)

In [ ]:
X_test, y_test = kpca_embedder.load_test_split("TD_extractor")

In [ ]:
print(X_test.shape)
print(y_test.shape)

In [ ]:
print(Z_train_64.shape)
print(y_train.shape)
print(Z_test_64.shape)
print(y_test.shape)

In [ ]:
## use a new loader
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm


class SafeECGLoader:
    def __init__(self, split_root):
        self.split_root = Path(split_root)

    def _get_shard_ids(self, modality):
       
        folder = self.split_root / modality

        files = sorted(folder.glob("*_shard_*.npz"))
        if not files:
            raise FileNotFoundError(f"No shards found in {folder}")

        shard_ids = []
        for f in files:
            # td_shard_003.npz → 003
            idx = int(f.stem.split("_")[-1])
            shard_ids.append(idx)

        return shard_ids

    def load_split(self, modality="time_domain_data", limit=None):
        """
        Returns:
            X: (N, D)
            y: (N,)
        """
        X_parts, y_parts = [], []

        shard_ids = self._get_shard_ids(modality)

        if limit is not None:
            shard_ids = shard_ids[:limit]

        for sid in tqdm(shard_ids, desc=f"Loading {modality}"):

            X_file = self.split_root / modality / f"td_shard_{sid:03d}.npz"
            y_file = self.split_root / "labels" / f"label_shard_{sid:03d}.npz"

            if not X_file.exists() or not y_file.exists():
                raise FileNotFoundError(f"Missing shard {sid}")

            X = np.load(X_file)["data"]
            y = np.load(y_file)["data"]

            if X.shape[0] != y.shape[0]:
                raise ValueError(
                    f"Mismatch in shard {sid}: X={X.shape}, y={y.shape}"
                )

            X_parts.append(X)
            y_parts.append(y)

        X_full = np.concatenate(X_parts, axis=0)
        y_full = np.concatenate(y_parts, axis=0)

        if X_full.shape[0] != y_full.shape[0]:
            raise ValueError("Global mismatch after concatenation")

        return X_full.astype(np.float32), y_full.astype(np.int64)

In [ ]:
## train  
train_loader = SafeECGLoader(r"C:\Users\wind2\ECGdata\processed\v1tts\train")

X_train, y_train = train_loader.load_split("time_domain_data")

In [ ]:
# test
test_loader = SafeECGLoader(r"C:\Users\wind2\ECGdata\processed\v1tts\test")

X_test, y_test = test_loader.load_split("time_domain_data")

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(
    n_components=64,
    kernel="rbf",
    gamma=0.01  # you can tune later
)

In [ ]:
Z_train = kpca.fit_transform(X_train_small)
Z_test  = kpca.transform(X_test_small)

In [ ]:
y_train_small = y_train[:5000]
y_test_small  = y_test[:2000]

In [ ]:
kpca_64_results = knn_eval(
    Z_train,
    y_train_small,
    Z_test,
    y_test_small
)

kpca_64_results

In [ ]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(
    n_components=64,
    kernel="rbf",
    gamma=0.01  # you can tune later
)

In [ ]:
def run_kpca_experiment(X_train, y_train, X_test, y_test, n_components):
    from sklearn.decomposition import KernelPCA

    kpca = KernelPCA(
        n_components=n_components,
        kernel="rbf",
        gamma=0.01
    )

    Z_train = kpca.fit_transform(X_train)
    Z_test = kpca.transform(X_test)

    return knn_eval(Z_train, y_train, Z_test, y_test)

In [ ]:
kpca_64_results = run_kpca_experiment(
    X_train_small, y_train_small,
    X_test_small, y_test_small,
    n_components=64
)

kpca_128_results = run_kpca_experiment(
    X_train_small, y_train_small,
    X_test_small, y_test_small,
    n_components=128
)

In [ ]:
import pandas as pd

kpca_table = pd.DataFrame({
    "K=1":  [kpca_64_results[1], kpca_128_results[1]],
    "K=5":  [kpca_64_results[5], kpca_128_results[5]],
    "K=10": [kpca_64_results[10], kpca_128_results[10]],
}, index=["KPCA-64", "KPCA-128"])

kpca_table

In [ ]:
## create all outputs

from pathlib import Path
import json
import numpy as np
from sklearn.decomposition import KernelPCA


def save_npz(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, data=array)


def run_kpca_experiment(
    X_train, y_train,
    X_test, y_test,
    n_components,
    run_name,
    output_root,
    kernel="rbf",
    gamma=0.01
):
    output_root = Path(output_root)
    run_dir = output_root / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # =========================
    # 1. KPCA embedding
    # =========================
    kpca = KernelPCA(
        n_components=n_components,
        kernel=kernel,
        gamma=gamma
    )

    Z_train = kpca.fit_transform(X_train)
    Z_test = kpca.transform(X_test)

    # =========================
    # 2. Save embeddings
    # =========================
    save_npz(run_dir / "train_embeddings.npz", Z_train)
    save_npz(run_dir / "test_embeddings.npz", Z_test)
    save_npz(run_dir / "train_labels.npz", y_train)
    save_npz(run_dir / "test_labels.npz", y_test)

    # KPCA equivalent of "projection components"
    if hasattr(kpca, "eigenvectors_"):
        save_npz(run_dir / "kpca_eigenvectors.npz", kpca.eigenvectors_)

    # =========================
    # 3. Save config
    # =========================
    config = {
        "method": "KernelPCA",
        "kernel": kernel,
        "gamma": gamma,
        "n_components": n_components,
        "train_size": len(X_train),
        "test_size": len(X_test)
    }

    with open(run_dir / "run_config.json", "w") as f:
        json.dump(config, f, indent=4)

    return Z_train, Z_test, run_dir

In [ ]:
Z_train_64, Z_test_64, dir_64 = run_kpca_experiment(
    X_train_small, y_train_small,
    X_test_small, y_test_small,
    n_components=64,
    run_name="kpca_td_rbf_64",
    output_root=r"C:\Users\wind2\ECGdata\outputs"
)

Z_train_128, Z_test_128, dir_128 = run_kpca_experiment(
    X_train_small, y_train_small,
    X_test_small, y_test_small,
    n_components=128,
    run_name="kpca_td_rbf_128",
    output_root=r"C:\Users\wind2\ECGdata\outputs"
)

In [ ]:
def evaluate_and_save(Z_train, y_train, Z_test, y_test, run_dir):
    import json
    import numpy as np

    results = {}
    predictions = {}

    for K in [1, 5, 10]:
        preds = knn_predict(Z_train, y_train, Z_test, K=K)
        preds = np.array(preds)

        acc = float((preds == y_test).mean())

        results[K] = acc
        predictions[K] = preds.tolist()

    # Save metrics
    with open(Path(run_dir) / "metrics.json", "w") as f:
        json.dump(results, f, indent=4)

    # Save predictions
    with open(Path(run_dir) / "predictions.json", "w") as f:
        json.dump(predictions, f, indent=4)

    return results

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier


def knn_predict(Z_train, y_train, Z_test, K=1):
    knn = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
    knn.fit(Z_train, y_train)
    return knn.predict(Z_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def knn_eval(Z_train, y_train, Z_test, y_test, K_values=[1,5,10]):
    results = {}

    for K in K_values:
        knn = KNeighborsClassifier(n_neighbors=K)
        knn.fit(Z_train, y_train)
        preds = knn.predict(Z_test)

        results[K] = float((preds == y_test).mean())

    return results

In [ ]:
kpca_64_results = evaluate_and_save(
    Z_train_64, y_train_small,
    Z_test_64, y_test_small,
    dir_64
)

kpca_128_results = evaluate_and_save(
    Z_train_128, y_train_small,
    Z_test_128, y_test_small,
    dir_128
)

In [ ]:
kpca_64_results

In [ ]:
kpca_128_results